In [2]:
!pip install -q selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.8/511.8 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 9.0 MB/s eta 0:00:00


In [4]:
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y -q ./google-chrome-stable_current_amd64.deb
!google-chrome --version

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-common at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0t64
  libatk1.0-0t64 libatspi2.0-0t64 libxcomposite1 libxtst6 session-migration
The following NEW packages will be installed:
  at-spi2-common at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0t64 libatk1.0-0t64 libatspi2.0-0t64 libxcomposite1
  libxtst6 session-migration
0 upgraded, 10 newly installed, 0 to remove and 0 not upgraded.
Need to get 331 kB/145 MB of archives.
After this operation, 467 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/main amd64 at-spi2-common all 2.52.0-1build1 [8,674 B]
Get:2 http://archive.ubuntu.com/ubuntu noble/main amd64 libatk1.0-0t64 amd64 2.52.0-1build1 [55.3 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble/main amd64 libatspi2.0-0t64 amd64 2.52.0-1build1 [80.5 kB]
Get:4 http:

In [5]:
HEADLESS = True

def make_driver():
    opts = Options()
    opts.binary_location = "/usr/bin/google-chrome"
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1400,1000")
    opts.add_argument("--lang=en-US")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument(
        "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(options=opts)

In [6]:
DATA_DIR = "/kaggle/input"

In [8]:
import glob
import json
import os
import random
import re
import shutil
import subprocess
import time
from datetime import date, timedelta
from urllib.parse import quote

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

# ----------------------------- Config -----------------------------
DATA_DIR = None             # None => download with kagglehub (works in Colab)
MAX_PER_CITY = 50           # set to 5 for a quick test
TEST_CITIES = None          # e.g. ["paris"] for a quick test; None => all cities
OUTPUT_CSV = "airbnb_listings_2br.csv"
CHECKIN = date.today() + timedelta(days=60)   # Airbnb needs dates to show prices
CHECKOUT = CHECKIN + timedelta(days=3)
ADULTS = 4

CITY_COUNTRY = {
    "amsterdam": "Netherlands", "athens": "Greece", "barcelona": "Spain",
    "berlin": "Germany", "budapest": "Hungary", "lisbon": "Portugal",
    "london": "United Kingdom", "paris": "France", "rome": "Italy",
    "vienna": "Austria",
}


# ------------------------- Environment setup -------------------------
def ensure_chrome():
    """Install Google Chrome if it's missing (Colab / Linux)."""
    if shutil.which("google-chrome"):
        return
    print("Installing Google Chrome ...")
    cmds = [
        "apt-get update -qq",
        "wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb",
        "apt-get install -y -qq ./google-chrome-stable_current_amd64.deb",
    ]
    for c in cmds:
        subprocess.run(c, shell=True, check=True)
    print(subprocess.run("google-chrome --version", shell=True,
                         capture_output=True, text=True).stdout)


def make_driver():
    opts = Options()
    opts.binary_location = shutil.which("google-chrome") or "/usr/bin/google-chrome"
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--window-size=1400,1000")
    opts.add_argument("--lang=en-US")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument(
        "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(options=opts)


# ------------------- Steps 1-2: read & filter data -------------------
def load_dataset() -> pd.DataFrame:
    folder = DATA_DIR
    if folder is None:
        import kagglehub
        folder = kagglehub.dataset_download("thedevastator/airbnb-prices-in-european-cities")
    frames = []
    for path in glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True):
        name = os.path.basename(path).lower()           # e.g. paris_weekdays.csv
        if "_weekdays" not in name and "_weekends" not in name:
            continue
        df = pd.read_csv(path)
        df["city"] = name.split("_")[0]
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


def pick_cities(df: pd.DataFrame) -> list:
    print("Cities in dataset:", sorted(df["city"].unique()))
    sub = df[(df["room_type"] == "Entire home/apt") & (df["bedrooms"] == 2)]
    counts = sub["city"].value_counts()
    print("Entire home/apt with 2 bedrooms per city:\n", counts)
    return sorted(counts.index.tolist())


# ------------------------- Selenium helpers -------------------------
def pause(a=1.5, b=3.5):
    time.sleep(random.uniform(a, b))


def dismiss_popups(driver):
    for xp in ["//button[contains(., 'Accept all')]", "//button[contains(., 'Accept')]",
               "//button[contains(., 'OK')]", "//button[@aria-label='Close']"]:
        try:
            driver.find_element(By.XPATH, xp).click()
            pause(0.5, 1)
        except Exception:
            pass


# ------------------ Steps 4-5: search page & links ------------------
def build_search_url(city: str) -> str:
    country = CITY_COUNTRY.get(city, "")
    place = quote(f"{city.title()}--{country}" if country else city.title())
    return (
        f"https://www.airbnb.com/s/{place}/homes"
        f"?room_types%5B%5D=Entire%20home%2Fapt"
        f"&min_bedrooms=2"
        f"&checkin={CHECKIN}&checkout={CHECKOUT}&adults={ADULTS}"
    )


def collect_links(driver, city: str, limit: int) -> list:
    driver.get(build_search_url(city))
    pause(4, 6)
    dismiss_popups(driver)
    print(f"[{city}] page title: {driver.title!r}")

    links, seen = [], set()
    for _page in range(6):
        for _ in range(6):                              # scroll to load lazy cards
            driver.execute_script("window.scrollBy(0, 900);")
            pause(0.6, 1.2)
        for a in driver.find_elements(By.CSS_SELECTOR, "a[href*='/rooms/']"):
            href = a.get_attribute("href") or ""
            m = re.search(r"/rooms/(\d+)", href)
            if m and m.group(1) not in seen:
                seen.add(m.group(1))
                links.append(
                    f"https://www.airbnb.com/rooms/{m.group(1)}"
                    f"?check_in={CHECKIN}&check_out={CHECKOUT}&adults={ADULTS}"
                )
        if len(links) >= limit:
            break
        try:
            driver.find_element(By.CSS_SELECTOR, "a[aria-label='Next']").click()
            pause(3, 5)
        except Exception:
            break

    if not links:
        # Diagnostics: probably blocked / CAPTCHA / different layout
        shot = f"debug_{city}.png"
        driver.save_screenshot(shot)
        print(f"[{city}] 0 links found. Screenshot saved to {shot}. "
              f"Likely blocked (Colab IPs are often blocked by Airbnb).")
    print(f"[{city}] collected {len(links)} links")
    return links[:limit]


# ----------------- Steps 6-7: listing page extraction -----------------
def extract_listing(driver, url: str, city: str) -> dict:
    driver.get(url)
    pause(3, 5)
    dismiss_popups(driver)
    for _ in range(3):
        driver.execute_script("window.scrollBy(0, 700);")
        pause(0.4, 0.8)

    text = driver.find_element(By.TAG_NAME, "body").text
    row = {"City": city.title(), "Property Name": None, "Price": None,
           "Rating": None, "Number of Reviews": None,
           "Superhost Status": False, "Guest Favorite Status": False,
           "Property URL": url.split("?")[0]}

    # Name
    try:
        row["Property Name"] = driver.find_element(By.TAG_NAME, "h1").text.strip()
    except Exception:
        pass

    # Rating / reviews: JSON-LD first
    for s in driver.find_elements(By.CSS_SELECTOR, "script[type='application/ld+json']"):
        try:
            data = json.loads(s.get_attribute("innerHTML"))
            agg = data.get("aggregateRating") if isinstance(data, dict) else None
            if agg:
                row["Rating"] = float(agg.get("ratingValue"))
                row["Number of Reviews"] = int(agg.get("ratingCount") or agg.get("reviewCount"))
                if not row["Property Name"]:
                    row["Property Name"] = data.get("name")
                break
        except Exception:
            continue

    # Fallback: regex on visible text
    if row["Rating"] is None:
        m = re.search(r"(\d\.\d{1,2})\s*[·•\n]\s*([\d,]+)\s+reviews?", text)
        if m:
            row["Rating"] = float(m.group(1))
            row["Number of Reviews"] = int(m.group(2).replace(",", ""))
        else:
            m = re.search(r"Rated\s+(\d\.\d{1,2})\s+out of 5", text)
            if m:
                row["Rating"] = float(m.group(1))
            m = re.search(r"([\d,]+)\s+reviews?", text)
            if m:
                row["Number of Reviews"] = int(m.group(1).replace(",", ""))
    if row["Number of Reviews"] is None and re.search(r"No reviews yet|\bNew\b", text):
        row["Number of Reviews"] = 0

    # Price
    m = re.search(r"([€$£]|EGP|EUR|USD|GBP)\s?([\d,]+(?:\.\d+)?)\s*(?:per\s+)?(night|total)?", text)
    if m:
        row["Price"] = f"{m.group(1)}{m.group(2)}" + (f" {m.group(3)}" if m.group(3) else "")

    # Badges
    row["Superhost Status"] = "Superhost" in text
    row["Guest Favorite Status"] = bool(re.search(r"Guest favou?rite", text, re.I))
    return row


# ------------------------------ Main ------------------------------
def main():
    df = load_dataset()
    cities = pick_cities(df)
    if TEST_CITIES:
        cities = [c for c in cities if c in TEST_CITIES]

    ensure_chrome()
    driver = make_driver()
    results = []
    try:
        for city in cities:
            try:
                links = collect_links(driver, city, MAX_PER_CITY)
            except Exception as e:
                print(f"[{city}] search failed: {e}")
                continue
            for i, url in enumerate(links, 1):
                try:
                    results.append(extract_listing(driver, url, city))
                    print(f"[{city}] {i}/{len(links)} OK")
                except Exception as e:
                    print(f"[{city}] {i} failed: {e}")
                pause(1.5, 3)
            pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    finally:
        driver.quit()

    out = pd.DataFrame(results)
    out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(out.shape)
    print(out.head())
    print(f"Saved to {OUTPUT_CSV}")
    return out


if __name__ == "__main__":
    # Quick test first: uncomment these two lines, then remove them for the full run
    # TEST_CITIES = ["paris"]; MAX_PER_CITY = 5
    result_df = main()
    # In Colab, to download the file:
    # from google.colab import files; files.download(OUTPUT_CSV)

Using Colab cache for faster access to the 'airbnb-prices-in-european-cities' dataset.
Cities in dataset: ['amsterdam', 'athens', 'barcelona', 'berlin', 'budapest', 'lisbon', 'london', 'paris', 'rome', 'vienna']
Entire home/apt with 2 bedrooms per city:
 city
rome         1756
athens       1489
london       1307
lisbon       1241
budapest      815
paris         719
vienna        556
amsterdam     354
barcelona     239
berlin        174
Name: count, dtype: int64
[amsterdam] page title: 'Airbnb | Amsterdam - Vacation Rentals & Places to Stay - North Holland'
[amsterdam] collected 18 links
[amsterdam] 1/18 OK
[amsterdam] 2/18 OK
[amsterdam] 3/18 OK
[amsterdam] 4/18 OK
[amsterdam] 5/18 OK
[amsterdam] 6/18 OK
[amsterdam] 7/18 OK
[amsterdam] 8/18 OK
[amsterdam] 9/18 OK
[amsterdam] 10/18 OK
[amsterdam] 11/18 OK
[amsterdam] 12/18 OK
[amsterdam] 13/18 OK
[amsterdam] 14/18 OK
[amsterdam] 15/18 OK
[amsterdam] 16/18 OK
[amsterdam] 17/18 OK
[amsterdam] 18/18 OK
[athens] page title: 'Airbnb | Athens